In [ ]:
using LinearAlgebra
using BoundaryValueDiffEq
using SparseArrays
using Plots
using BenchmarkTools
using ModelingToolkit
using DifferentialEquations
using DomainSets

In [ ]:
@parameters gamma, gamma_3, c, omega, x, t
@variables A(..), B(..)
r1 = @rule cos(~x)^3 => 0.75 * cos(~x) + 0.25 * cos(3 * ~x)
r2 = @rule sin(~x)^3 => 0.75 * sin(~x) - 0.25 * sin(3 * ~x)
r3 = @rule cos(~x)^2 => 1 - sin(~x)^2
r4 = @rule sin(~x)^2 => 1 - cos(~x)^2

u = A(x) * sin(omega*t) # + B(x) * cos(omega*t)
Dx = Differential(x)
Dt = Differential(t)
y = Dt(Dt(u)) - c*c*Dx(Dx(u)) + gamma*Dt(u) # + gamma_3*Dt(u)*Dt(u)*Dt(u)
y_exp = expand_derivatives(y)

y_exp = simplify(expand(y_exp), RuleSet([r1, r2, r3, r4]))
y_exp = expand(y_exp)
y_exp = simplify(y_exp, RuleSet([r1, r2, r3, r4]))

sin_coeff = Symbolics.coeff(y_exp, sin(omega*t))

sin_sol = Symbolics.solve_for(sin_coeff ~ 0, Differential(x)(Differential(x)(A(x))))
println("Sin:")
println("Sin coeff: ", sin_coeff)
println("Axx = ", sin_sol)

f = Symbolics.build_function(sin_sol, A(x), omega, c; expression=Val{false})

In [ ]:
# set source function 
src(x) = 1000. # obtain u(x) > 1 for [u(x)}^3 to be large in size 

# set right-hand side 
# u''(x) = - f(x) reformulated as 
# coupled first order system as 
# u'(x) = v(x) and v'(x) = - f(x) or 
# du[1] = u[2] and du[2] = - f(x) 
function rhs!(du, u, p, x)
    c, omd = p
    B = u[1]; dB = u[2]
    du[1] = dB
    du[2] = f(B, omd, c) - src(x)/c 
end

# set boundary conditions 
# boundary conditions on the left and right of computational domain 
# requires explaining syntax 
function bc(residual, u, p, x)
    uleft = 0; uright=0; 
    residual[1] = u[1][1] - uleft 
    residual[2] = u[end][1] - uright
end

In [ ]:
# set constants 
c = 1.
gamma = 1. 
gamma3 = 0  
omd = 10.
p = [c, omd]

omdstep = 1. 
omdvec  = 0.:omdstep:10. |> collect 

# set domain size 
xspan = (0.,1.)

# set initial guess 
start = [0., 0.]

# store parameter value in p  
p = [c, omd]
    
# define and solve the problem 
bvp = BVProblem(rhs!, bc, start, xspan,p)
sol = solve(bvp)

In [ ]:
# plot u(x) and du/dx(x)
myB  = [u[1] for u in sol.u]
mydB = [u[2] for u in sol.u]
p1 = plot(sol.t, omd^2/c^2*myB, xlabel="x", ylabel = "omd^2/c^2*u(x)") 
p2 = plot(sol.t, mydB, xlabel="x", ylabel = "du(x)") 
plot(p1,p2,layout=(1,2))

In [ ]:
Nt = 1
dt = 0.01
u = [x[1] for x in sol.u] 
animation = @animate for k in 1:(Nt/dt)
    new_u = u * sin(omd * (k * dt)) 
    plot(sol.t, new_u,
         title="t = $(k * dt)", xlabel="x", ylabel="u(x,t)", ylim = (-50, 50))
end

gif(animation, "Example_harmonic.gif", fps = 10)